# SOLAR V25 — FilamentGraph-PQ | Dynamic multi-branch research suite

**Standalone Kaggle notebook, not an existing 0.40 score.** Preserves the exact V22 0.37 detector recipe and the **V24 quality-0.25 control (public PQ 0.38)**, then evaluates separately: (A) cross-fitted extended quality ranker, (B) trainable 3-head curve / centerline / boundary branch, (C) proposal-driven refiner fine-tuning, (D) learned fragment linking. Generates five or more independently labeled submissions and saves checkpoints.

### Inputs — attach two Kaggle datasets
1. The **official `solar3k`** dataset (707 training / 180 test + JSON).
2. Your **V24 output archive**, ideally `SOLAR_V24_TRAINED_MODELS_AND_SUBMISSIONS.zip` (contains the original V20 YOLO11m detectors, `SOLAR_V24_REFINER_BEST.pt`, `SOLAR_V24_INSTANCE_QUALITY.joblib`, and CSV/report archive). Alternatively attach those four files directly. The **V24 quality model is required to reproduce the known 0.38 method**; this notebook does not silently pretend a different model is that control.

T4 GPU (T4×2 uses GPU 0), Internet ON if Kaggle has to install missing dependencies. Run All once; checkpoint outputs can be reattached later. For a Kaggle runtime limit, adjust the bounded budgets in cell 1. The notebook can optionally train OOF detector folds, but that is expensive and is **off by default**. Without that option, cross-fitting is applied to the new ranking model across the old 104 held-out detector scenes; it does not make the repeatedly reused V20 validation dataset genuinely untouched.

**Research integrity:** official train labels only, no external test labels, no leaderboard-label inference. The new proposal branch is experimental; each module is measured before selection. Keep your separately submitted V24 0.38 as the historical benchmark even if local output selection differs.

In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); OUT=Path('/kaggle/working/solar_v25')
SEED=2026; GPU_ID=0; H=W=2048
RAW_CONF=.08; RAW_IOU=.72; MAX_DET=120
ALLOW_NON_V20_WEIGHT_HASH=False
V24_REFERENCE_SCORE=.38
V24_EXPECTED_REFERENCE_SHA='1f26c3989955da1e'  # published V24 QUALITY_025 CSV
CROP_SIDE=320; REFINE_PRED_THRESHOLD=.50; REFINEMENT_MIN_MASK_IOU=.54
# On 1 T4, training a compact branch and real-proposal finetune is realistic.
CURVE_TRAIN_SCENES=270; CURVE_CROPS_PER_SCENE=3; CURVE_SIDE=512
CURVE_EPOCHS=9; CURVE_BATCH=6; CURVE_STEPS_PER_EPOCH=165
CURVE_INFER=True; CURVE_WINDOW=768; CURVE_STRIDE=640
CURVE_MIN_AREA=170; CURVE_MAX_PROPOSALS=42
REAL_REFINE_TRAIN_SCENES=115; REAL_REFINE_MAX_PAIRS=900
REAL_REFINE_EPOCHS=4; REAL_REFINE_BATCH=8; REAL_REFINE_STEPS_PER_EPOCH=130
RUN_CURVE=True; RUN_REAL_REFINER=True; RUN_GRAPH=True; RUN_NEW_QUALITY=True
# Require entire historical validation set and both V24 model assets. No silent downgrade.
FULL_VALIDATION_REQUIRED=True
REQUIRE_V24_MODELS=True
MAX_TEST_ALTERNATIVES=6
for d in ('models','raw','refined','reports','submissions','input_data','curve_bank','curve_raw','real_pairs','checkpoints','graph'):
    (OUT/d).mkdir(parents=True,exist_ok=True)
assert INPUT.is_dir(), 'Kaggle input not found'
print('V25 output root:',OUT)

In [ ]:
import os,sys,time,json,hashlib,zipfile,shutil,gzip,random,subprocess,importlib.util
import gc as _gc_safe
from collections import defaultdict
import numpy as np,pandas as pd,cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from IPython.display import display

def require(module,package):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
require('pycocotools','pycocotools')
# Lock exact V22 version when available to maximize baseline reproduction.
try:
    from importlib.metadata import version as package_version
    installed_version=package_version('ultralytics')
except Exception:
    installed_version=None
if installed_version!='8.4.189':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.189'])
from pycocotools import mask as coco
from ultralytics import YOLO
import ultralytics,torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU T4/T4x2. No CPU or nano fallback.'
assert torch.cuda.device_count()>GPU_ID
print('Using',torch.cuda.get_device_name(GPU_ID),'ultralytics',ultralytics.__version__)
cv2.setNumThreads(0);random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
H=W=2048
EXT={'.jpeg','.jpg','.png','.tif','.tiff','.bmp'}
def files_in(d):
    p=Path(d)
    return sorted(q for q in p.iterdir() if q.is_file() and q.suffix.lower() in EXT) if p.is_dir() else []
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def save_json(p,obj):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_suffix(p.suffix+'.tmp');temp.write_text(json.dumps(obj,indent=2,default=str));temp.replace(p)
def enc(m):return coco.encode(np.asfortranarray(np.asarray(m,np.uint8)))
def txt(r):return r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else str(r['counts'])
def as_rle(x):return {'size':[H,W],'counts':x.encode('ascii') if isinstance(x,str) else x}
assert callable(_gc_safe.collect)
save_json(OUT/'reports'/'env.json',{'torch':torch.__version__,'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(GPU_ID),'no_prior_split_required':True,'prior_verified_public_PQ':0.37,'target_public_PQ':0.40})

## 1. Official input and historical date-group split

In [ ]:
ANAME='MAGFiLO_1.0_Annotations_kaggle2026_train.json'
found=sorted(INPUT.rglob(ANAME))
OFFICIAL=next((p.parent.parent for p in found if (p.parent.parent/'test'/'test_images').is_dir() and (p.parent.parent/'train'/'train_images').is_dir()),None)

def find_zip(name):
    matches=list(INPUT.rglob(name))
    return matches[0] if matches else None

def safe_images_unzip(archive,destination):
    destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            if not item.filename.lower().endswith(tuple(EXT)):continue
            target=destination/Path(item.filename).name
            if not target.exists():
                with z.open(item) as src,open(target,'wb') as dst:shutil.copyfileobj(src,dst,1<<20)
    return destination
if OFFICIAL:
    ANNOT=OFFICIAL/'train'/ANAME
    TRAIN={p.stem:p for p in files_in(OFFICIAL/'train'/'train_images')}
    TEST={p.stem:p for p in files_in(OFFICIAL/'test'/'test_images')}
    print('Using full official competition folder',OFFICIAL)
else:
    annot=found[0] if found else None
    ztest=find_zip('test.zip');ztrain=find_zip('sometrain.zip')
    assert annot is not None and ztest is not None, 'Attach official solar3k, or the annotation JSON and test.zip as Kaggle inputs.'
    ANNOT=annot
    TEST={p.stem:p for p in files_in(safe_images_unzip(ztest,OUT/'input_data'/'test_images'))}
    TRAIN={p.stem:p for p in files_in(safe_images_unzip(ztrain,OUT/'input_data'/'train_images'))} if ztrain else {}
    print('Using upload ZIP fallback. Validation may cover fewer than 104 physical images.')
assert len(TEST)==180,('Expected 180 test scenes; found',len(TEST))
assert len(set(TEST)&set(TRAIN))==0
DOC=json.loads(ANNOT.read_text())
BY_ID=defaultdict(list)
for ann in DOC['annotations']:BY_ID[str(ann['image_id'])].append(ann)
VERSIONS=defaultdict(list)
for item in DOC['images']:
    stem=Path(item['file_name']).stem
    if stem in TRAIN:VERSIONS[stem].append((str(item['id']),BY_ID.get(str(item['id']),[])))
assert set(TRAIN)==set(VERSIONS)
if TRAIN:
    days=defaultdict(list)
    for img in DOC['images']:
        stem=Path(img['file_name']).stem
        days[stem[:8]].append(stem)
    days={k:sorted(set(v)) for k,v in days.items()}
    shuffled=sorted(days)
    random.Random(SEED+2020).shuffle(shuffled)
    selected_days=[];n=0
    for d in shuffled:
        if n>=104:break
        selected_days.append(d);n+=len(days[d])
    VALIDATION=sorted(set(stem for d in selected_days for stem in days[d]) & set(TRAIN))
else:VALIDATION=[]
FULL_V20_VALIDATION=(len(VALIDATION)==104 and len(TRAIN)==707)
print('Images: train present',len(TRAIN),'test',len(TEST),'V20 tune available',len(VALIDATION),'full reproduction',FULL_V20_VALIDATION)
if VALIDATION:
    assert not ({s[:8] for s in VALIDATION} & {s[:8] for s in TRAIN if s not in VALIDATION})
save_json(OUT/'reports'/'dataset.json',{'annotation_sha256':sha(ANNOT),'train_available':len(TRAIN),
    'test_images':len(TEST),'validation_available':len(VALIDATION),'full_v20_reproduction':FULL_V20_VALIDATION,
    'validation_stems':VALIDATION,'test_ground_truth_used':False})

## 2. Restore verified trained V20 detectors (not pretrained-only weights)

In [ ]:
EXPECT={
 'portable_global_m':('portable_global_m_best.pt','73fd3f3a4931'),
 'portable_native_m':('portable_native_m_best.pt','1a932d28e590'),
}
V20_DIR=Path('/kaggle/input/datasets/lighittha/solarouts/solar_v20_from_scratch/models')
ZIPS=sorted(set(p for p in INPUT.rglob('*.zip') if any(token in p.name.lower() for token in ('v20','v21','weights','models'))))
WEIGHTS={};HASHES={};SOURCES={}
(OUT/'models').mkdir(parents=True,exist_ok=True)
for model_name,(filename,prefix) in EXPECT.items():
    path=OUT/'models'/filename
    choices=[]
    direct=V20_DIR/filename
    if direct.is_file():choices.append(direct)
    choices+=[p for p in INPUT.rglob(filename) if p.is_file() and p!=direct]
    if path.is_file():choices.insert(0,path)
    if not path.is_file() or path.stat().st_size<20_000_000:
        restored=False
        for src in choices:
            if 30_000_000<src.stat().st_size<65_000_000:
                if src!=path:shutil.copy2(src,path)
                SOURCES[model_name]=str(src);restored=True;break
        if not restored:
            for archive in ZIPS:
                try:
                    with zipfile.ZipFile(archive) as z:
                        member='models/'+filename
                        if member not in z.namelist():continue
                        if not (30_000_000<z.getinfo(member).file_size<65_000_000):continue
                        temp=path.with_suffix('.part')
                        with z.open(member) as rf,open(temp,'wb') as wf:shutil.copyfileobj(rf,wf,1<<20)
                        temp.replace(path);SOURCES[model_name]=str(archive)+'::'+member
                        restored=True;break
                except (OSError,zipfile.BadZipFile) as e:print('Ignore corrupt ZIP:',archive,e)
        if not restored:raise FileNotFoundError('Missing trained V20 '+filename+f'. Expected at {V20_DIR/filename} or inside the V20 weights ZIP. This notebook never substitutes pretrained-only or nano models.')
    digest=sha(path)
    if not ALLOW_NON_V20_WEIGHT_HASH:
        assert digest.startswith(prefix),f'{filename}: wrong weights hash! Expected V20 prefix {prefix}, got {digest[:12]}'
    m=YOLO(str(path))
    n_params=sum(p.numel() for p in m.model.parameters())
    assert m.task=='segment' and 21_000_000<=n_params<=24_000_000,(model_name,m.task,n_params)
    del m;_gc_safe.collect()
    WEIGHTS[model_name]=path;HASHES[model_name]=digest
print('Recovered V20 model identity:',{k:v[:12] for k,v in HASHES.items()})
save_json(OUT/'reports'/'weights.json',{'sha256':HASHES,'source':SOURCES,'V20_checkpoint_hash_gate':not ALLOW_NON_V20_WEIGHT_HASH})

## 3. Locked 0.37 detector recipe, original 1792-global / native 1280 tiles

In [ ]:
# Locked V22 successful source definitions.
SRC_BASE_GLOBAL=('portable_global_m','g1536')
SRC_BASE_NATIVE=('portable_native_m','g1536')
SRC_HI=('portable_global_m','g1792')
SRC_TILES=('portable_native_m','t1280')
SRC_HIFLIP=('portable_global_m','g1792_hflip')
BASE_SOURCES=[SRC_HI,SRC_TILES]
EXTRA_SOURCES=[]
ACTIVE=BASE_SOURCES+EXTRA_SOURCES
ENGINES={}
def get_engine(name):
    if name not in ENGINES:
        ENGINES.clear();_gc_safe.collect();torch.cuda.empty_cache()
        ENGINES[name]=YOLO(str(WEIGHTS[name]))
    return ENGINES[name]

def view_windows(view):
    if view=='g1536':return [(0,0,2048,1536,False)]
    if view=='g1792':return [(0,0,2048,1792,False)]
    if view=='g1792_hflip':return [(0,0,2048,1792,True)]
    if view=='t1280':return [(x,y,1280,1024,False) for y in (0,768) for x in (0,768)]
    raise ValueError(view)
def sig(source):
    name,view=source
    return hashlib.sha256(json.dumps({'model_sha256':HASHES[name],'view':view,
      'conf':RAW_CONF,'iou':RAW_IOU,'max_det':MAX_DET,'retina_masks':True,
      'agnostic_nms':True,'ultralytics':ultralytics.__version__,'version':'V22_1' if view in ('g1792','t1280','g1536') else 'V24_1'},sort_keys=True).encode()).hexdigest()
def cache_file(source,stem):
    name,view=source
    return OUT/'raw'/name/view/(stem+'.json.gz')
@torch.inference_mode()
def raw_predictions(source,stem):
    path=cache_file(source,stem);identifier=sig(source)
    if path.exists():
        try:
            with gzip.open(path,'rt') as f:v=json.load(f)
            if v.get('signature')==identifier and v.get('stem')==stem:return v['predictions']
        except (OSError,ValueError,KeyError):pass
    image_path=TRAIN.get(stem) or TEST.get(stem)
    if image_path is None:raise FileNotFoundError('Missing scene: '+stem)
    gray=cv2.imread(str(image_path),cv2.IMREAD_GRAYSCALE)
    assert gray is not None and gray.shape==(H,W),(stem, None if gray is None else gray.shape)
    predictions=[]
    name,view=source
    model=get_engine(name)
    for x,y,side,net,hflip in view_windows(view):
        roi=np.ascontiguousarray(gray[y:y+side,x:x+side])
        if hflip:roi=cv2.flip(roi,1)
        image=cv2.cvtColor(roi,cv2.COLOR_GRAY2BGR)
        result=model.predict(image,imgsz=net,conf=RAW_CONF,iou=RAW_IOU,max_det=MAX_DET,
                             agnostic_nms=True,retina_masks=True,device=GPU_ID,verbose=False)[0]
        if result.masks is None or result.boxes is None:continue
        mask_data=result.masks.data.detach().cpu().numpy()
        confidences=result.boxes.conf.detach().cpu().numpy()
        for mask_arr,conf in zip(mask_data,confidences):
            mask=(mask_arr>0.5).astype(np.uint8)
            if mask.shape!=(side,side):mask=cv2.resize(mask,(side,side),interpolation=cv2.INTER_NEAREST)
            if hflip:mask=np.ascontiguousarray(cv2.flip(mask,1))
            if int(mask.sum())<100:continue
            if side!=2048:
                ys,xs=np.where(mask)
                truncated=bool((x>0 and xs.min()<10) or (x+side<W and xs.max()>=side-10) or
                               (y>0 and ys.min()<10) or (y+side<H and ys.max()>=side-10))
                full=np.zeros((H,W),np.uint8)
                full[y:y+side,x:x+side]=mask
            else:truncated=False;full=mask
            rr=enc(full)
            predictions.append({'counts':txt(rr),'score':float(conf),'area':int(coco.area(rr)),
                                'truncated':truncated,'source_view':view})
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with gzip.open(tmp,'wt',compresslevel=3) as f:
        json.dump({'signature':identifier,'stem':stem,'view':view,'predictions':predictions},f,separators=(',',':'))
    tmp.replace(path)
    return predictions
print('V23 views including immutable winner:',ACTIVE)

In [ ]:
def recipe(name,sources,gconf=.24,tconf=.32,area=200,seam=False,refine='none',minfree=.45):
    return {'name':name,'sources':[list(s) for s in sources],'global_conf':float(gconf),
            'tile_conf':float(tconf),'min_area':int(area),'seam_guard':bool(seam),
            'nms_iou':.50,'tile_bonus':.08,'min_free':float(minfree),'refine':refine}

# This exact configuration produced V22_ALT_2.csv (public PQ 0.37).
WINNER=recipe('V22_ALT2_WINNER_037',[SRC_HI,SRC_TILES],
              gconf=.24,tconf=.42,area=200,seam=True,refine='none',minfree=.45)
WINNER['nms_iou']=.50
WINNER['tile_bonus']=.08
WINNER['engine']='v22'
WINNER['fusion']='none'
WINNER['tile_policy']='baseline'

def experiment(name,sources=None,gconf=.24,tconf=.42,area=200,seam=True,
               bonus=.08,nms=.50,minfree=.45,tile_policy='baseline',fusion='none'):
    r=recipe(name,sources if sources is not None else [SRC_HI,SRC_TILES],
             gconf=gconf,tconf=tconf,area=area,seam=seam,minfree=minfree)
    r.update(engine='v23',tile_bonus=bonus,nms_iou=nms,tile_policy=tile_policy,fusion=fusion)
    return r


def modify_rle(rr,operation):
    if operation=='none':return rr
    box=coco.toBbox(rr);x,y,w,h=[int(q) for q in box]
    if w<1 or h<1:return rr
    pad=3;x0=max(0,x-pad);y0=max(0,y-pad);x1=min(W,x+w+pad);y1=min(H,y+h+pad)
    full=coco.decode(rr).astype(np.uint8)
    roi=full[y0:y1,x0:x1]
    if operation=='close3':
        result=cv2.morphologyEx(roi,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
    elif operation=='dilate1':
        result=cv2.dilate(roi,np.ones((3,3),np.uint8),iterations=1)
    elif operation=='erode1':
        result=cv2.erode(roi,np.ones((3,3),np.uint8),iterations=1)
    else:raise ValueError(operation)
    if int(result.sum())<50:return rr
    full[y0:y1,x0:x1]=result
    return enc(full)

def select_masks(raw,cfg):
    sources={tuple(pair) for pair in cfg['sources']}
    candidates=[]
    for source,items in raw.items():
        if source not in sources:continue
        threshold=cfg['tile_conf'] if source[1].startswith('t') else cfg['global_conf']
        for p in items:
            score=float(p['score']);area=int(p['area'])
            if score<threshold or area<cfg['min_area']:continue
            if cfg['seam_guard'] and source[1].startswith('t') and p.get('truncated',False):continue
            rank=score+(cfg['tile_bonus'] if source[1].startswith('t') else 0.)
            candidates.append((rank,area,p))
    candidates.sort(key=lambda q:(-q[0],-q[1]))
    masks=[];originals=[];occupied=np.zeros((H,W),np.uint8)
    for rank,area,p in candidates:
        rr=as_rle(p['counts'])
        if originals:
            ov=np.asarray(coco.iou([rr],originals,[0]*len(originals)))[0]
            if np.any(ov>=cfg['nms_iou']):continue
        mask=coco.decode(rr).astype(np.uint8)
        free=mask&(occupied==0)
        free_area=int(free.sum())
        if free_area<cfg['min_area'] or free_area/max(1,area)<cfg['min_free']:continue
        cleaned=enc(free)
        if cfg.get('refine','none')!='none':cleaned=modify_rle(cleaned,cfg['refine'])
        # Retain exact V20 occupied behavior for baseline; for refined recipes use actual refined footprint.
        if cfg.get('refine','none')!='none':
            shaped=coco.decode(cleaned).astype(np.uint8)&(occupied==0)
            if int(shaped.sum())<cfg['min_area']:continue
            cleaned=enc(shaped)
            occupied|=shaped
        else:occupied|=free
        masks.append(cleaned);originals.append(rr)
        if len(masks)>=120:break
    return masks

# Do not change the winner branch; these tests protect the baseline.
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['global_conf']==.24 and WINNER['tile_conf']==.42
assert WINNER['seam_guard'] and WINNER['min_area']==200
assert WINNER['nms_iou']==.5 and WINNER['tile_bonus']==.08
assert callable(_gc_safe.collect)

## 4. Local PQ scorer (historical proxy, not guaranteed identical to private evaluator)

In [ ]:
def annotation_rle(a):
    seg=a.get('segmentation')
    if seg is None or seg==[]:return None
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):
            obj=coco.frPyObjects(seg,H,W)
            return coco.merge(obj) if isinstance(obj,list) else obj
        rr=dict(seg)
        if isinstance(rr.get('counts'),str):rr['counts']=rr['counts'].encode('ascii')
        return rr
    if len(seg)>0 and isinstance(seg[0],(int,float)):seg=[seg]
    polygons=[p for p in seg if len(p)>=6 and len(p)%2==0]
    if not polygons:return None
    rr=coco.frPyObjects(polygons,H,W)
    return coco.merge(rr) if isinstance(rr,list) else rr
TRUTH={}
def gt(stem):
    if stem not in TRUTH:
        TRUTH[stem]=[[rr for a in arr if (rr:=annotation_rle(a)) is not None]
                     for _,arr in VERSIONS[stem]]
    return TRUTH[stem]
def match_counts(truth,preds):
    if not truth:return np.array([0.,0.,float(len(preds)),0.])
    if not preds:return np.array([0.,0.,0.,float(len(truth))])
    ious=np.asarray(coco.iou(truth,preds,[0]*len(preds)),float)
    # Match the earlier V20 PQ proxy EXACTLY for control-score parity.
    ii,jj=linear_sum_assignment(-ious)
    ok=ious[ii,jj]>.5
    tp=int(ok.sum())
    return np.array([float(ious[ii[ok],jj[ok]].sum()),float(tp),
                     float(len(preds)-tp),float(len(truth)-tp)])
def physical_counts(stem,preds):
    result=np.zeros(4,float)
    for version in gt(stem):result+=match_counts(version,preds)
    return result
def pq(count):
    den=count[1]+.5*(count[2]+count[3])
    return float(count[0]/den) if den>0 else 0.
def metrics(count):
    den=count[1]+.5*(count[2]+count[3])
    return {'PQ':pq(count),'SQ':float(count[0]/count[1]) if count[1] else 0.,
       'RQ':float(count[1]/den) if den else 0.,'TP':int(count[1]),
       'FP':int(count[2]),'FN':int(count[3])}
example=np.zeros((H,W),np.uint8);example[40:61,50:92]=1
assert coco.area(as_rle(txt(enc(example))))==example.sum()
assert abs(pq(match_counts([enc(example)],[enc(example)]))-1)<1e-9
print('PQ scorer and full-size RLE roundtrip: PASS')

## 5. Recover the actual trained V24 refiner and reliability model

This is needed to preserve the 0.38 public method. V24 masks may be precomputed in the user's saved files, but predictions here are regenerated from the exact V20 weights plus the frozen V24 artifacts. All proposed V25 modules are trained **separately**.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import joblib, math, time, io
from torchvision.models import resnet18

class DoubleConv(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.block=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
            nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x):return self.block(x)

class PromptUNet(nn.Module):
    def __init__(self):
        super().__init__();e=resnet18(weights=None);old=e.conv1
        e.conv1=nn.Conv2d(4,64,7,2,3,bias=False)
        self.stem=nn.Sequential(e.conv1,e.bn1,e.relu);self.pool=e.maxpool
        self.e1=e.layer1;self.e2=e.layer2;self.e3=e.layer3;self.e4=e.layer4
        self.d4=DoubleConv(768,192);self.d3=DoubleConv(320,128)
        self.d2=DoubleConv(192,96);self.d1=DoubleConv(160,64)
        self.out=nn.Sequential(nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                               nn.Conv2d(64,32,3,padding=1),nn.ReLU(inplace=True),nn.Conv2d(32,1,1))
    def forward(self,x):
        e0=self.stem(x);e1=self.e1(self.pool(e0));e2=self.e2(e1);e3=self.e3(e2);e4=self.e4(e3)
        def up(a,b):return torch.cat([F.interpolate(a,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1)
        a=self.d4(up(e4,e3));a=self.d3(up(a,e2));a=self.d2(up(a,e1));a=self.d1(up(a,e0))
        return self.out(a)

def make_channels(gray_uint8,mask_uint8):
    gray=np.asarray(gray_uint8,dtype=np.uint8)
    equal=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(gray)
    blur=cv2.GaussianBlur(gray,(3,3),0)
    dx=cv2.Sobel(blur,cv2.CV_32F,1,0,ksize=3);dy=cv2.Sobel(blur,cv2.CV_32F,0,1,ksize=3)
    edge=np.clip(cv2.magnitude(dx,dy)/240.,0,1)
    return np.stack([gray/255.,equal/255.,mask_uint8.astype(np.float32),edge],axis=0).astype(np.float32)

def normalized_bbox_roi(box,scale=1.48,min_extent=120):
    x,y,w,h=map(float,box)
    if w<=0 or h<=0:return None
    dx=min(W,max(float(min_extent),w*scale));dy=min(H,max(float(min_extent),h*scale))
    cx=x+w/2;cy=y+h/2
    x0=int(np.clip(round(cx-dx/2),0,W-1));y0=int(np.clip(round(cy-dy/2),0,H-1))
    x1=int(np.clip(round(cx+dx/2),x0+1,W));y1=int(np.clip(round(cy+dy/2),y0+1,H))
    return x0,y0,x1,y1

def find_original_asset(filename):
    direct=sorted(INPUT.rglob(filename))
    for p in direct:
        if p.is_file() and p.stat().st_size>100:return p.read_bytes(),str(p)
    for z in sorted(INPUT.rglob('*.zip')):
        if not any(token in z.name.lower() for token in ('v24','model','trained')):continue
        try:
            with zipfile.ZipFile(z) as arc:
                matches=[m for m in arc.namelist() if Path(m).name==filename]
                if matches:return arc.read(matches[0]),str(z)+'::'+matches[0]
                # V24 full output bundles a second ZIP containing its scored CSVs.
                for nested in [m for m in arc.namelist() if m.lower().endswith('.zip')]:
                    try:
                        with zipfile.ZipFile(io.BytesIO(arc.read(nested))) as inner:
                            hits=[x for x in inner.namelist() if Path(x).name==filename]
                            if hits:return inner.read(hits[0]),str(z)+'::'+nested+'::'+hits[0]
                    except (OSError,zipfile.BadZipFile):pass
        except (OSError,zipfile.BadZipFile):continue
    return None,None

for filename in ('SOLAR_V24_REFINER_BEST.pt','SOLAR_V24_INSTANCE_QUALITY.joblib'):
    target=OUT/'models'/filename
    data,source=find_original_asset(filename)
    if data is None:
        if REQUIRE_V24_MODELS:raise FileNotFoundError('Attach V24 model archive; missing '+filename)
    else:
        if not target.exists() or sha(target)!=hashlib.sha256(data).hexdigest():target.write_bytes(data)
        print('Recovered V24 asset:',filename, 'from',source,'SHA:',sha(target)[:16])

V24_REFINER_PATH=OUT/'models'/'SOLAR_V24_REFINER_BEST.pt'
V24_QUALITY_PATH=OUT/'models'/'SOLAR_V24_INSTANCE_QUALITY.joblib'
refiner_v24=PromptUNet().to('cuda:'+str(GPU_ID))
refiner_v24.load_state_dict(torch.load(V24_REFINER_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_v24.eval()
meta_v24=joblib.load(V24_QUALITY_PATH)
quality_v24=meta_v24['model'];V24_FEATURES=meta_v24['features']
assert len(V24_FEATURES)==15 and callable(quality_v24.predict_proba)
print('Recovered genuine V24 quality model, OOF AUROC:',meta_v24.get('oof_auc'))

## 6. Reconstruct V24 control and exact 15-feature predictor

In [ ]:
def matching_scores(rr,records):
    if not records:return 0.,0.,0.
    rles=[as_rle(q['counts']) for q in records]
    ious=np.asarray(coco.iou([rr],rles,[0]*len(rles)),float).reshape(-1)
    i=int(ious.argmax());return float(ious[i]),float(records[i]['score']),float(max(q['score'] for q in records))

@torch.inference_mode()
def run_refiner(model,gray,rr,threshold=.50,min_prompt_iou=.54):
    crop=normalized_bbox_roi([int(v) for v in coco.toBbox(rr)],scale=1.48)
    if crop is None:return rr
    x0,y0,x1,y1=crop;original=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    if not original.any():return rr
    image=cv2.resize(gray[y0:y1,x0:x1],(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_AREA)
    prompt=cv2.resize(original,(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_NEAREST)
    inp=torch.from_numpy(make_channels(image,prompt))[None].cuda(GPU_ID)
    with torch.autocast('cuda',dtype=torch.float16):
        probabilities=model(inp).sigmoid()[0,0].float().cpu().numpy()
    refined=(cv2.resize(probabilities,(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)>=threshold).astype(np.uint8)
    oa=int(original.sum());na=int(refined.sum())
    if na<80 or not (.52<=na/max(oa,1)<=1.65):return rr
    overlap=int((refined&original).sum())/max(1,oa+na-int((refined&original).sum()))
    if overlap<min_prompt_iou:return rr
    full=np.zeros((H,W),np.uint8);full[y0:y1,x0:x1]=refined
    return enc(full)

def load_base(stem):
    raw={SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    return raw,select_masks(raw,WINNER)

def v24_refined_masks(stem,baseline,gray=None):
    key=sha(V24_REFINER_PATH)[:18];p=OUT/'refined'/('v24_'+key+'_'+stem+'.json.gz')
    if p.is_file():
        with gzip.open(p,'rt') as f:v=json.load(f)
        if len(v)==len(baseline):return [as_rle(x) for x in v]
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    proposed=[run_refiner(refiner_v24,gray,rr) for rr in baseline]
    occupied=np.zeros((H,W),np.uint8);final=[]
    for a,b in zip(baseline,proposed):
        f=coco.decode(b).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:f=coco.decode(a).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:continue
        final.append(enc(f));occupied|=f
    if len(final)!=len(baseline):final=baseline
    p.parent.mkdir(parents=True,exist_ok=True)
    with gzip.open(p,'wt',compresslevel=3) as f:json.dump([txt(v) for v in final],f)
    return final

def features_v24(stem,base,refined,gray=None,raw=None):
    raw=raw if raw is not None else {SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    rows=[]
    for rr,new in zip(base,refined):
        x,y,w,h=map(float,coco.toBbox(rr));mask=coco.decode(rr).astype(np.uint8);area=int(mask.sum())
        dilation=cv2.dilate(mask,np.ones((11,11),np.uint8))
        ring=(dilation>0)&(mask==0)
        inside=float(gray[mask>0].mean()) if area else 0.;outside=float(gray[ring].mean()) if ring.any() else inside
        ga,gscore,gmax=matching_scores(rr,raw[SRC_HI]);ta,tscore,tmax=matching_scores(rr,raw[SRC_TILES])
        na=float(coco.area(new));oa=float(coco.area(rr))
        riou=float(coco.iou([rr],[new],[0])[0,0])
        rows.append({'base_score':max(gscore,tscore),'base_area_log':float(np.log1p(area)),
                     'base_box_ar':float(np.log((w+1)/(h+1))),'base_fill':float(area/max(1,w*h)),
                     'base_center_x':float((x+w/2)/W),'base_center_y':float((y+h/2)/H),
                     'global_agreement':ga,'tile_agreement':ta,'global_maxconf':gmax,'tile_maxconf':tmax,
                     'pixel_avg_inside':inside/255.,'pixel_avg_ring':outside/255.,
                     'contrast':(outside-inside)/255.,'refined_iou':riou,'refined_area_ratio':na/max(oa,1.)})
    return pd.DataFrame(rows,columns=V24_FEATURES)

def v24_control(stem):
    raw,base=load_base(stem)
    if not base:return []
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    refined=v24_refined_masks(stem,base,gray)
    df=features_v24(stem,base,refined,gray,raw)
    qs=quality_v24.predict_proba(df[V24_FEATURES].fillna(0.))[:,1]
    return [rr for rr,q in zip(base,qs) if q>=.25]  # Exact V24 QUALITY_025: no extra clipping

s=np.zeros((H,W),np.uint8);s[50:90,70:130]=1
assert int(coco.area(enc(s)))==2400
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['tile_conf']==.42 and WINNER['global_conf']==.24 and WINNER['seam_guard']
print('V24 control frozen. New V25 scoring cannot alter its thresholds or masks.')

## 7. Train a *new* curve-specialist foreground / centerline / boundary branch

A compact 3-head U-Net is trained on filament-containing windows sampled from **603 historical detector-training scenes**, using annotation-version sampling. It predicts pixelwise filament probability, centerline probability and boundary probability. Inference later extracts candidate instances and offers only nonduplicate ones to the learned quality scorer. Neither the V24 quality control nor any test labels are part of this training.

In [ ]:
require('skimage','scikit-image')
from skimage.morphology import skeletonize
from skimage.segmentation import watershed
CURVE_BANK=OUT/'curve_bank';(CURVE_BANK/'im').mkdir(parents=True,exist_ok=True)
(CURVE_BANK/'gt').mkdir(parents=True,exist_ok=True)
CURVE_INDEX=CURVE_BANK/'index.csv'

def crop_annotated_mask(annot,x0,y0,side):
    seg=annot.get('segmentation',[])
    out=np.zeros((side,side),np.uint8)
    if isinstance(seg,dict):
        rr=annotation_rle(annot)
        if rr is not None:out=coco.decode(rr)[y0:y0+side,x0:x0+side].astype(np.uint8)
    else:
        if seg and isinstance(seg[0],(float,int)):seg=[seg]
        for p in seg:
            if len(p)<6 or len(p)%2:continue
            pts=np.asarray(p,np.float32).reshape(-1,2)-[x0,y0]
            cv2.fillPoly(out,[np.rint(pts).astype(np.int32)],1)
    return out

curve_records=pd.read_csv(CURVE_INDEX).to_dict('records') if CURVE_INDEX.is_file() else []
curve_known={r['key'] for r in curve_records}
train_stems=sorted(set(TRAIN)-set(VALIDATION))
assert len(train_stems)==603
rng_scene=random.Random(SEED+725);rng_scene.shuffle(train_stems)
for stem in tqdm(train_stems[:CURVE_TRAIN_SCENES],desc='Build curve-specialist training bank'):
    gray=cv2.imread(str(TRAIN[stem]),cv2.IMREAD_GRAYSCALE)
    versions=VERSIONS[stem];valid_versions=[v for _,v in versions if len(v)]
    if not valid_versions:continue
    rng=random.Random(SEED+int(hashlib.sha256(stem.encode()).hexdigest()[:7],16))
    for j in range(CURVE_CROPS_PER_SCENE):
        key=f'{stem}_window{j}'
        if key in curve_known:continue
        anns=rng.choice(valid_versions)
        anchor=rng.choice(anns)
        bx,by,bw,bh=map(float,anchor.get('bbox',[0,0,0,0]))
        cx=bx+bw/2+rng.uniform(-160,160);cy=by+bh/2+rng.uniform(-160,160)
        x0=int(np.clip(cx-CURVE_WINDOW/2,0,W-CURVE_WINDOW));y0=int(np.clip(cy-CURVE_WINDOW/2,0,H-CURVE_WINDOW))
        roi=gray[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]
        # Same one expert version for all objects in the window; not an artificial union of disagreeing labels.
        foreground=np.zeros((CURVE_WINDOW,CURVE_WINDOW),np.uint8)
        for a in anns:foreground|=crop_annotated_mask(a,x0,y0,CURVE_WINDOW)
        if foreground.sum()<150:continue
        im_small=cv2.resize(roi,(CURVE_SIDE,CURVE_SIDE),interpolation=cv2.INTER_AREA)
        fg_small=(cv2.resize(foreground,(CURVE_SIDE,CURVE_SIDE),interpolation=cv2.INTER_AREA)>.25).astype(np.uint8)
        if fg_small.sum()<70:continue
        cv2.imwrite(str(CURVE_BANK/'im'/(key+'.png')),im_small)
        cv2.imwrite(str(CURVE_BANK/'gt'/(key+'.png')),fg_small*255)
        curve_records.append({'key':key,'stem':stem,'x':x0,'y':y0,'annotations':len(anns)})
        curve_known.add(key)
    if len(curve_records)%100<CURVE_CROPS_PER_SCENE:
        pd.DataFrame(curve_records).to_csv(CURVE_INDEX,index=False)
pd.DataFrame(curve_records).to_csv(CURVE_INDEX,index=False)
assert len(curve_records)>150,('Curve bank too small',len(curve_records))
print('Curve bank:',len(curve_records),'crops across',len({r['stem'] for r in curve_records}),'scenes')

class CurveCrops(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        key=self.rows[i]['key'];im=cv2.imread(str(CURVE_BANK/'im'/(key+'.png')),0)
        fg=cv2.imread(str(CURVE_BANK/'gt'/(key+'.png')),0)
        if im is None or fg is None:raise FileNotFoundError(key)
        fg=(fg>127).astype(np.uint8)
        if self.augment:
            if np.random.rand()<.5:im=np.fliplr(im).copy();fg=np.fliplr(fg).copy()
            if np.random.rand()<.5:im=np.flipud(im).copy();fg=np.flipud(fg).copy()
            if np.random.rand()<.5:
                k=int(np.random.randint(4));im=np.rot90(im,k).copy();fg=np.rot90(fg,k).copy()
        center=skeletonize(fg>0).astype(np.float32)
        border=(cv2.morphologyEx(fg,cv2.MORPH_GRADIENT,np.ones((3,3),np.uint8))>0).astype(np.float32)
        equ=cv2.createCLAHE(clipLimit=2.,tileGridSize=(8,8)).apply(im)
        inp=np.stack([im/255.,equ/255.],0).astype(np.float32)
        target=np.stack([fg.astype(np.float32),center,border],0)
        return torch.from_numpy(inp),torch.from_numpy(target)

class MiniUNet(nn.Module):
    def __init__(self,channels=2,outputs=3):
        super().__init__()
        self.a=DoubleConv(channels,24);self.b=DoubleConv(24,48);self.c=DoubleConv(48,96)
        self.d=DoubleConv(96,160)
        self.up3=DoubleConv(160+96,96);self.up2=DoubleConv(96+48,64)
        self.up1=DoubleConv(64+24,32);self.final=nn.Conv2d(32,outputs,1)
    def forward(self,x):
        a=self.a(x);b=self.b(F.avg_pool2d(a,2));c=self.c(F.avg_pool2d(b,2));d=self.d(F.avg_pool2d(c,2))
        u=self.up3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode='bilinear',align_corners=False),c],1))
        u=self.up2(torch.cat([F.interpolate(u,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        u=self.up1(torch.cat([F.interpolate(u,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return self.final(u)

def curve_loss(logits,target):
    weights=torch.tensor([1.,2.5,1.8],device=logits.device).view(1,3,1,1)
    elem=F.binary_cross_entropy_with_logits(logits,target,reduction='none')
    pos=torch.tensor([1.,7.,4.],device=logits.device).view(1,3,1,1)
    bce=((1+target*(pos-1))*elem*weights).mean()
    probs=torch.sigmoid(logits[:,0]);y=target[:,0]
    dice=1-(2*(probs*y).sum((1,2))+1)/((probs+y).sum((1,2))+1)
    return .28*bce+.72*dice.mean()

curve_path=Path('/kaggle/working/SOLAR_V25_CURVE_BEST.pt')
curve_last=Path('/kaggle/working/SOLAR_V25_CURVE_LAST.pt')
if not curve_path.is_file():
    saved_bytes,origin=find_original_asset(curve_path.name)
    if saved_bytes is not None:
        curve_path.write_bytes(saved_bytes)
        print('Restored trained curve model from',origin)
curve_net=MiniUNet().cuda(GPU_ID)
# The scene holdout is within the 603 training scenes, distinct from the 104 detector validation scenes.
scene_curve=sorted({r['stem'] for r in curve_records});random.Random(SEED+509).shuffle(scene_curve)
curve_hold=set(scene_curve[:max(25,int(len(scene_curve)*.13))])
curve_train=[r for r in curve_records if r['stem'] not in curve_hold]
curve_val=[r for r in curve_records if r['stem'] in curve_hold]
curve_tloader=DataLoader(CurveCrops(curve_train,True),batch_size=CURVE_BATCH,shuffle=True,
                         num_workers=2,pin_memory=True)
curve_vloader=DataLoader(CurveCrops(curve_val),batch_size=CURVE_BATCH,shuffle=False,
                         num_workers=2,pin_memory=True)
curve_optim=torch.optim.AdamW(curve_net.parameters(),lr=2e-4,weight_decay=2e-4)
curve_scaler=torch.amp.GradScaler('cuda')
if curve_path.exists():
    curve_ckpt=torch.load(curve_path,map_location='cpu',weights_only=False)
    curve_net.load_state_dict(curve_ckpt['model']);curve_start=curve_ckpt['epoch']+1
    curve_best=float(curve_ckpt['val_loss'])
else:curve_start=0;curve_best=float('inf')

if RUN_CURVE:
    for epoch in range(curve_start,CURVE_EPOCHS):
        curve_net.train();tr_losses=[]
        for i,(x,y) in enumerate(curve_tloader):
            if i>=CURVE_STEPS_PER_EPOCH:break
            x=x.cuda(GPU_ID,non_blocking=True);y=y.cuda(GPU_ID,non_blocking=True)
            curve_optim.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):loss=curve_loss(curve_net(x),y)
            curve_scaler.scale(loss).backward();curve_scaler.step(curve_optim);curve_scaler.update()
            tr_losses.append(float(loss.detach()))
        curve_net.eval();vals=[]
        with torch.no_grad():
            for i,(x,y) in enumerate(curve_vloader):
                if i>=45:break
                x=x.cuda(GPU_ID);y=y.cuda(GPU_ID)
                with torch.autocast('cuda',dtype=torch.float16):vals.append(float(curve_loss(curve_net(x),y)))
        metric=float(np.mean(vals)) if vals else float(np.mean(tr_losses))
        package={'model':{k:v.cpu() for k,v in curve_net.state_dict().items()},'epoch':epoch,
                 'val_loss':metric,'train_scenes':len(set(r['stem'] for r in curve_train)),
                 'no_test_labels':True,'heads':['foreground','centerline','boundary']}
        torch.save(package,curve_last)
        if metric<curve_best:curve_best=metric;torch.save(package,curve_path)
        print('Curve epoch',epoch,'loss',round(np.mean(tr_losses),4),'val',round(metric,4),'best',round(curve_best,4),flush=True)
else:
    assert curve_path.exists(),'RUN_CURVE=False requires attached curve checkpoint'
assert curve_path.is_file(),'Curve branch did not produce a checkpoint'
curve_net.load_state_dict(torch.load(curve_path,map_location='cpu',weights_only=False)['model'])
curve_net.eval();CURVE_SHA=sha(curve_path)
print('Curve checkpoint ready SHA:',CURVE_SHA[:18])

## 8. Extract **new** curve instances, rather than relabeling the old YOLO masks

Use overlapping tiles, per-instance connectivity analysis and optional watershed splitting. We cache all raw candidate RLEs. A conservative duplicate-and-darkness check controls proposal proliferation; the final quality classifier determines which new instances to keep.

In [ ]:
def curve_locations():
    starts=list(range(0,H-CURVE_WINDOW+1,CURVE_STRIDE))
    if starts[-1]!=H-CURVE_WINDOW:starts.append(H-CURVE_WINDOW)
    return [(x,y) for y in starts for x in starts]
CURVE_LOCATIONS=curve_locations()
assert len(CURVE_LOCATIONS)==9, CURVE_LOCATIONS

def curve_input(gray):
    equ=cv2.createCLAHE(clipLimit=2.,tileGridSize=(8,8)).apply(gray)
    return torch.from_numpy(np.stack([gray/255.,equ/255.],0).astype(np.float32))[None].cuda(GPU_ID)

@torch.inference_mode()
def curve_proposals(stem):
    cache=OUT/'curve_raw'/(stem+'.json.gz')
    signature=hashlib.sha256((CURVE_SHA+stem+str(CURVE_WINDOW)+str(CURVE_SIDE)).encode()).hexdigest()
    if cache.exists():
        try:
            with gzip.open(cache,'rt') as f:v=json.load(f)
            if v.get('identity')==signature:return v['proposals']
        except (ValueError,OSError,KeyError):pass
    image=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    if image is None:raise FileNotFoundError(stem)
    props=[]
    for x0,y0 in CURVE_LOCATIONS:
        raw=image[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]
        inp=cv2.resize(raw,(CURVE_SIDE,CURVE_SIDE),interpolation=cv2.INTER_AREA)
        with torch.autocast('cuda',dtype=torch.float16):
            probs=curve_net(curve_input(inp)).sigmoid()[0].float().cpu().numpy()
        fg=cv2.resize(probs[0],(CURVE_WINDOW,CURVE_WINDOW))
        center=cv2.resize(probs[1],(CURVE_WINDOW,CURVE_WINDOW))
        seg=(fg>.53).astype(np.uint8)
        n,components,stats,_=cv2.connectedComponentsWithStats(seg,connectivity=8)
        for k in range(1,n):
            component_area=int(stats[k,cv2.CC_STAT_AREA])
            if component_area<CURVE_MIN_AREA:continue
            local=(components==k).astype(np.uint8)
            # Conservatively split only when two *distinct* centerline islands exist.
            line=(center>.56).astype(np.uint8)&local
            m,markers,mark_stats,_=cv2.connectedComponentsWithStats(line,connectivity=8)
            good=[q for q in range(1,m) if mark_stats[q,cv2.CC_STAT_AREA]>=8]
            pieces=[local]
            if 2<=len(good)<=5 and component_area>700:
                seeds=np.zeros_like(local,np.int32)
                for label_idx,mark_idx in enumerate(good,1):seeds[markers==mark_idx]=label_idx
                regions=watershed(-cv2.distanceTransform(local,cv2.DIST_L2,3),seeds,mask=(local>0))
                candidates=[(regions==z).astype(np.uint8) for z in range(1,len(good)+1)]
                if all(int(c.sum())>=CURVE_MIN_AREA for c in candidates):pieces=candidates
            for part in pieces:
                area=int(part.sum())
                if area<CURVE_MIN_AREA:continue
                sy,sx=np.where(part)
                if min(sx)<8 and x0>0 or max(sx)>CURVE_WINDOW-9 and x0+CURVE_WINDOW<W:continue
                if min(sy)<8 and y0>0 or max(sy)>CURVE_WINDOW-9 and y0+CURVE_WINDOW<H:continue
                local_gray=raw[part>0];mask_score=float(np.mean(fg[part>0]))
                line_score=float(np.mean(center[part>0]))
                if mask_score<.59 or line_score<.18:continue
                # Relatively dark features, not arbitrary bright solar active-region structures.
                patch=raw[max(0,sy.min()-16):min(CURVE_WINDOW,sy.max()+17),
                          max(0,sx.min()-16):min(CURVE_WINDOW,sx.max()+17)]
                if len(local_gray) and float(np.median(local_gray))>float(np.percentile(patch,75))+8:continue
                full=np.zeros((H,W),np.uint8)
                full[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]=part
                rr=enc(full)
                props.append({'counts':txt(rr),'score':float(np.clip(.65*mask_score+.35*line_score,0,1)),
                              'area':int(coco.area(rr)),'truncated':False,'source_view':'curve_v25',
                              'center_support':line_score})
    props.sort(key=lambda p:p['score'],reverse=True)
    distinct=[];rles=[]
    for p in props:
        rr=as_rle(p['counts'])
        if rles and float(np.max(coco.iou([rr],rles,[0]*len(rles))))>.33:continue
        distinct.append(p);rles.append(rr)
        if len(distinct)>=CURVE_MAX_PROPOSALS:break
    with gzip.open(cache,'wt',compresslevel=3) as f:
        json.dump({'identity':signature,'stem':stem,'proposals':distinct},f,separators=(',',':'))
    return distinct

def extend_with_curve(stem,baseline,raw=None):
    # Existing instance identities are never silently split or replaced here.
    raw=raw if raw is not None else load_base(stem)[0]
    base_rles=list(baseline)
    base_all=base_rles.copy()
    occupied=np.zeros((H,W),np.uint8)
    for rr in base_rles:occupied|=coco.decode(rr).astype(np.uint8)
    additions=[]
    for p in curve_proposals(stem):
        rr=as_rle(p['counts'])
        if base_all and float(np.max(coco.iou([rr],base_all,[0]*len(base_all))))>=.25:continue
        part=coco.decode(rr).astype(np.uint8)
        free=part&(occupied==0)
        if int(free.sum())<CURVE_MIN_AREA or free.sum()/max(1,part.sum())<.70:continue
        accepted=enc(free)
        additions.append(accepted);occupied|=free;base_all.append(accepted)
    return base_rles+additions,['detector']*len(base_rles)+['curve']*len(additions)

print('Curve candidate generation initialized:',len(CURVE_LOCATIONS),'windows per 2048 scene')

## 9. Fine-tune the refiner on **real detector errors** (not synthetic prompts)

Frozen V20 models make proposals on a deterministic subset of their original training scenes. We match proposals to every available expert version and train with genuine partial/irregular masks. The new refiner is separately checkpointed; the V24 checkpoint remains unchanged and controls the 0.38 baseline. A scene-separated holdout selects the best new refiner checkpoint. For an even stronger study, detector out-of-fold proposals should replace in-sample proposals; the limitation is recorded.

In [ ]:
REAL_BANK=OUT/'real_pairs';(REAL_BANK/'im').mkdir(exist_ok=True)
(REAL_BANK/'prompt').mkdir(exist_ok=True);(REAL_BANK/'target').mkdir(exist_ok=True)
real_index=REAL_BANK/'index.csv';existing_real=pd.read_csv(real_index).to_dict('records') if real_index.exists() else []
known_real={r['key'] for r in existing_real};real_rows=list(existing_real)
selected_real=sorted(set(TRAIN)-set(VALIDATION));random.Random(SEED+881).shuffle(selected_real)

for stem in tqdm(selected_real[:REAL_REFINE_TRAIN_SCENES],desc='Build genuine YOLO→expert refiner pairs'):
    if len(real_rows)>=REAL_REFINE_MAX_PAIRS:break
    raw,proposals=load_base(stem)
    if not proposals:continue
    gray=cv2.imread(str(TRAIN[stem]),0)
    expert_rles=[r for version in gt(stem) for r in version]
    if not expert_rles:continue
    iou_mat=np.asarray(coco.iou(proposals,expert_rles,[0]*len(expert_rles)),float)
    for j,rr in enumerate(proposals):
        if len(real_rows)>=REAL_REFINE_MAX_PAIRS:break
        key=f'{stem}_{j}'
        if key in known_real:continue
        best_idx=int(np.argmax(iou_mat[j]));best_iou=float(iou_mat[j,best_idx])
        # Refine localization errors, and a modest sample of hard negatives.
        label_positive=best_iou>=.22
        if not label_positive and not (best_iou<.08 and j%8==0):continue
        roi=normalized_bbox_roi(coco.toBbox(rr),scale=1.48)
        if roi is None:continue
        x0,y0,x1,y1=roi
        coarse=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
        if label_positive:target=coco.decode(expert_rles[best_idx])[y0:y1,x0:x1].astype(np.uint8)
        else:target=np.zeros((y1-y0,x1-x0),np.uint8)
        img=cv2.resize(gray[y0:y1,x0:x1],(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_AREA)
        pr=cv2.resize(coarse,(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_NEAREST)
        gt_small=cv2.resize(target,(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(str(REAL_BANK/'im'/(key+'.jpg')),img,[cv2.IMWRITE_JPEG_QUALITY,95])
        cv2.imwrite(str(REAL_BANK/'prompt'/(key+'.png')),pr*255)
        cv2.imwrite(str(REAL_BANK/'target'/(key+'.png')),gt_small*255)
        real_rows.append({'key':key,'stem':stem,'matched_iou':best_iou,'positive':int(label_positive)})
        known_real.add(key)
    if len(real_rows)%100<14:pd.DataFrame(real_rows).to_csv(real_index,index=False)
pd.DataFrame(real_rows).to_csv(real_index,index=False)
assert len(real_rows)>90,('Too few real detector supervision pairs',len(real_rows))
print('Genuine detector-prompt pairs:',len(real_rows),'positive fraction:',round(np.mean([r['positive'] for r in real_rows]),3))

class DetectorPairs(Dataset):
    def __init__(self,rows,augment):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        key=self.rows[i]['key'];g=cv2.imread(str(REAL_BANK/'im'/(key+'.jpg')),0)
        m=cv2.imread(str(REAL_BANK/'prompt'/(key+'.png')),0)
        t=cv2.imread(str(REAL_BANK/'target'/(key+'.png')),0)
        if any(v is None for v in (g,m,t)):raise FileNotFoundError(key)
        m=(m>127).astype(np.uint8);t=(t>127).astype(np.uint8)
        if self.augment:
            if np.random.rand()<.5:g=np.fliplr(g).copy();m=np.fliplr(m).copy();t=np.fliplr(t).copy()
            if np.random.rand()<.5:g=np.flipud(g).copy();m=np.flipud(m).copy();t=np.flipud(t).copy()
        return torch.from_numpy(make_channels(g,m)),torch.from_numpy(t[None].astype(np.float32))

def real_seg_loss(logit,target):
    bce=F.binary_cross_entropy_with_logits(logit,target)
    pr=logit.sigmoid();num=2*(pr*target).sum((2,3))+1
    den=(pr+target).sum((2,3))+1
    dice=(1-num/den).mean()
    return .5*bce+.5*dice

real_scene_ids=sorted({r['stem'] for r in real_rows});random.Random(SEED+112).shuffle(real_scene_ids)
real_hold=set(real_scene_ids[:max(12,int(len(real_scene_ids)*.15))])
real_train=[r for r in real_rows if r['stem'] not in real_hold]
real_valid=[r for r in real_rows if r['stem'] in real_hold]
assert len(real_train)>50 and len(real_valid)>10
refiner_real=PromptUNet().cuda(GPU_ID)
real_best_path=Path('/kaggle/working/SOLAR_V25_REAL_REFINER_BEST.pt')
real_last_path=Path('/kaggle/working/SOLAR_V25_REAL_REFINER_LAST.pt')
if not real_best_path.is_file():
    saved_bytes,origin=find_original_asset(real_best_path.name)
    if saved_bytes is not None:
        real_best_path.write_bytes(saved_bytes)
        print('Restored trained real-prompt refiner from',origin)
if real_best_path.is_file():
    ckpt=torch.load(real_best_path,map_location='cpu',weights_only=False)
    refiner_real.load_state_dict(ckpt['model']);real_start=ckpt['epoch']+1;real_best=float(ckpt['val_loss'])
else:
    refiner_real.load_state_dict(torch.load(V24_REFINER_PATH,map_location='cpu',weights_only=False)['model'])
    real_start=0;real_best=float('inf')
real_optim=torch.optim.AdamW(refiner_real.parameters(),lr=4e-5,weight_decay=1e-4)
real_scaler=torch.amp.GradScaler('cuda')
real_train_loader=DataLoader(DetectorPairs(real_train,True),batch_size=REAL_REFINE_BATCH,shuffle=True,
                             num_workers=2,pin_memory=True)
real_val_loader=DataLoader(DetectorPairs(real_valid,False),batch_size=REAL_REFINE_BATCH,shuffle=False,
                           num_workers=2,pin_memory=True)
if RUN_REAL_REFINER:
    for epoch in range(real_start,REAL_REFINE_EPOCHS):
        refiner_real.train();losses=[]
        for idx,(x,y) in enumerate(real_train_loader):
            if idx>=REAL_REFINE_STEPS_PER_EPOCH:break
            x=x.cuda(GPU_ID);y=y.cuda(GPU_ID)
            real_optim.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):loss=real_seg_loss(refiner_real(x),y)
            real_scaler.scale(loss).backward();real_scaler.step(real_optim);real_scaler.update()
            losses.append(float(loss.detach()))
        refiner_real.eval();values=[]
        with torch.inference_mode():
            for i,(x,y) in enumerate(real_val_loader):
                if i>=35:break
                x=x.cuda(GPU_ID);y=y.cuda(GPU_ID)
                with torch.autocast('cuda',dtype=torch.float16):values.append(float(real_seg_loss(refiner_real(x),y)))
        metric=float(np.mean(values))
        ckpt={'model':{k:v.detach().cpu() for k,v in refiner_real.state_dict().items()},
              'epoch':epoch,'val_loss':metric,'real_detector_prompts':True,'test_labels_used':False}
        torch.save(ckpt,real_last_path)
        if metric<real_best:real_best=metric;torch.save(ckpt,real_best_path)
        print('Real-prompt refiner epoch',epoch,'train',round(np.mean(losses),4),'val',round(metric,4),'best',round(real_best,4))
else:assert real_best_path.exists(),'Attach V25 refiner checkpoint if training disabled'
assert real_best_path.exists()
refiner_real.load_state_dict(torch.load(real_best_path,map_location='cpu',weights_only=False)['model'])
refiner_real.eval();REAL_REFINER_SHA=sha(real_best_path)
print('Actual-detector-prompt refiner:',REAL_REFINER_SHA[:16])

def real_refine(stem,rr,gray=None):
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
    return run_refiner(refiner_real,gray,rr,threshold=.5,min_prompt_iou=.50)

## 10. Extended instance reliability representation: morphology, image context, sources, topology

We retain all V24 features and add scale, shape, skeleton, curvature proxies, local image statistics, model agreement, solar-disk location and neighbor features. The output schema is deterministic and shared by training/validation/test. Unlike the V24 15-feature model, this learns from curve proposals too. **Model-selection decisions use date-separated OOF scores and a 51-scene audit, not test labels.**

In [ ]:
from scipy.spatial.distance import cdist

# These 40+ new features are computed from *predicted masks/images only*.
EXTRA_FIELDS=['log_perimeter','log_convex_area','solidity','extent','elongation','eccentricity',
 'aspect_absolute','area_frac_box','edge_density','endpoint_count','branchpoint_count','skeleton_length',
 'skeleton_area_ratio','mean_thickness','max_thickness','width_variation','local_component_count',
 'intensity_inside_std','intensity_inside_p10','intensity_inside_p50','intensity_inside_p90',
 'intensity_ring_p50','intensity_diff_p10','intensity_diff_mean','inside_entropy','sobel_inside_mean',
 'sobel_ring_mean','contrast_normalized','sun_radial','rim_distance','mask_near_solar_limb',
 'hull_vs_mask','compactness','hu_1','hu_2','pca_angle_sin','pca_angle_cos',
 'neighbor_center_dist','neighbor_bbox_iou','neighbor_area_ratio','n_neighbors_100',
 'curve_origin','curve_conf','curve_center_support','curve_vs_global_iou','curve_vs_tile_iou',
 'raw_global_count','raw_tile_count','raw_high_conf_global_count','raw_high_conf_tile_count',
 'mask_global_rank','mask_tile_rank','mask_area_over_disk','intensity_outside_std']
NEW_FEATURES=list(V24_FEATURES)+EXTRA_FIELDS
assert len(NEW_FEATURES)>60

def image_entropy(pixels):
    if len(pixels)==0:return 0.
    hist=np.bincount((pixels//16).astype(np.uint8),minlength=16).astype(float)
    hist=hist/max(hist.sum(),1);h=hist[hist>0]
    return float(-(h*np.log(h)).sum())

def extract_extra(gray,rr,all_boxes,source_kind,curve_meta,raw):
    x,y,w,h=map(float,coco.toBbox(rr));x0=max(0,int(x)-12);y0=max(0,int(y)-12)
    x1=min(W,int(np.ceil(x+w))+12);y1=min(H,int(np.ceil(y+h))+12)
    # COCO decode is full-frame but geometric operations are limited to a small crop.
    mask=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    g=gray[y0:y1,x0:x1];area=int(mask.sum());yy,xx=np.where(mask>0)
    ring=cv2.dilate(mask,np.ones((9,9),np.uint8))&(1-mask)
    inside=g[mask>0];outside=g[ring>0]
    if not len(inside):inside=np.asarray([0],np.uint8)
    if not len(outside):outside=inside
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    perimeter=float(sum(cv2.arcLength(c,True) for c in contours))
    hull_area=float(sum(cv2.contourArea(cv2.convexHull(c)) for c in contours if len(c)>=3))
    # Downsample skeleton for speed; normalize pixel-to-image ratio.
    ratio=min(1.,150/max(mask.shape));sw=max(8,int(mask.shape[1]*ratio));sh=max(8,int(mask.shape[0]*ratio))
    reduced=(cv2.resize(mask,(sw,sh),interpolation=cv2.INTER_NEAREST)>0).astype(np.uint8)
    sk=skeletonize(reduced>0).astype(np.uint8)
    neigh=cv2.filter2D(sk,cv2.CV_16S,np.ones((3,3),np.uint8),borderType=cv2.BORDER_CONSTANT)-sk.astype(np.int16)
    endpoints=int(((sk>0)&(neigh==1)).sum());branches=int(((sk>0)&(neigh>=3)).sum())
    dist=cv2.distanceTransform(reduced,cv2.DIST_L2,3)[reduced>0]
    components=cv2.connectedComponents(reduced,connectivity=8)[0]-1
    if len(xx)>1:
        cov=np.cov(np.stack([xx,yy]).astype(float));ev,vec=np.linalg.eigh(cov)
        major=vec[:,np.argmax(ev)];elong=float(np.sqrt((max(ev)+1)/(min(ev)+1)))
        ecc=float(np.sqrt(max(0.,1-min(ev)/(max(ev)+1e-6))))
    else:major=np.array([1.,0.]);elong=1.;ecc=0.
    sob=cv2.magnitude(cv2.Sobel(g,cv2.CV_32F,1,0),cv2.Sobel(g,cv2.CV_32F,0,1))
    hu=cv2.HuMoments(cv2.moments(mask)).flatten()
    cx=x+w/2;cy=y+h/2;rad=np.hypot(cx-W/2,cy-H/2)/(W/2)
    other_boxes=np.asarray(all_boxes,float).reshape(-1,4)
    current_centroid=np.array([cx,cy]);centroids=other_boxes[:,:2]+other_boxes[:,2:]/2 if len(other_boxes) else np.zeros((0,2))
    distances=np.linalg.norm(centroids-current_centroid,axis=1) if len(centroids) else np.asarray([])
    nearby=sorted(float(z) for z in distances if z>1)
    bbox_ious=[]
    for bx,by,bw,bh in other_boxes:
        if abs(bx-x)+abs(by-y)+abs(bw-w)+abs(bh-h)<.001:continue
        iw=max(0,min(x+w,bx+bw)-max(x,bx));ih=max(0,min(y+h,by+bh)-max(y,by))
        bbox_ious.append(iw*ih/max(1,w*h+bw*bh-iw*ih))
    raw_g=raw[SRC_HI];raw_t=raw[SRC_TILES]
    gi,gc,gmax=matching_scores(rr,raw_g);ti,tc,tmax=matching_scores(rr,raw_t)
    flat=dict(log_perimeter=np.log1p(perimeter),log_convex_area=np.log1p(hull_area),
        solidity=area/max(1,hull_area),extent=area/max(1,w*h),elongation=min(100,elong),
        eccentricity=ecc,aspect_absolute=max((w+1)/(h+1),(h+1)/(w+1)),area_frac_box=area/max(1,w*h),
        edge_density=perimeter/max(1,area),endpoint_count=endpoints,branchpoint_count=branches,
        skeleton_length=int(sk.sum())/max(.15,ratio),skeleton_area_ratio=float(sk.sum())/max(1,reduced.sum()),
        mean_thickness=float(np.mean(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        max_thickness=float(np.max(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        width_variation=float(np.std(dist)) if len(dist) else 0.,local_component_count=max(0,components),
        intensity_inside_std=float(inside.std())/255.,intensity_inside_p10=float(np.percentile(inside,10))/255.,
        intensity_inside_p50=float(np.median(inside))/255.,intensity_inside_p90=float(np.percentile(inside,90))/255.,
        intensity_ring_p50=float(np.median(outside))/255.,
        intensity_diff_p10=float(np.percentile(outside,10)-np.percentile(inside,10))/255.,
        intensity_diff_mean=float(outside.mean()-inside.mean())/255.,inside_entropy=image_entropy(inside),
        sobel_inside_mean=float(sob[mask>0].mean())/255. if area else 0.,
        sobel_ring_mean=float(sob[ring>0].mean())/255. if ring.any() else 0.,
        contrast_normalized=float(outside.mean()-inside.mean())/max(10.,float(outside.std())),
        sun_radial=rad,rim_distance=max(0,1-rad),mask_near_solar_limb=float(rad>.93),
        hull_vs_mask=hull_area/max(1,area),compactness=perimeter**2/max(1,4*np.pi*area),
        hu_1=float(np.sign(hu[0])*np.log1p(abs(hu[0]))),hu_2=float(np.sign(hu[1])*np.log1p(abs(hu[1]))),
        pca_angle_sin=float(major[1]),pca_angle_cos=float(major[0]),
        neighbor_center_dist=nearby[0]/2048 if nearby else 1.,
        neighbor_bbox_iou=max(bbox_ious,default=0.),
        neighbor_area_ratio=float(sorted([abs(np.log((w*h+1)/(bw*bh+1))) for bx,by,bw,bh in other_boxes])[1]) if len(other_boxes)>1 else 0.,
        n_neighbors_100=sum(v<100 for v in nearby),curve_origin=float(source_kind=='curve'),
        curve_conf=float(curve_meta.get('score',0)),curve_center_support=float(curve_meta.get('center_support',0)),
        curve_vs_global_iou=gi,curve_vs_tile_iou=ti,
        raw_global_count=len(raw_g),raw_tile_count=len(raw_t),
        raw_high_conf_global_count=sum(p['score']>.4 for p in raw_g),
        raw_high_conf_tile_count=sum(p['score']>.4 for p in raw_t),
        mask_global_rank=gc,mask_tile_rank=tc,mask_area_over_disk=area/(np.pi*1024**2),
        intensity_outside_std=float(outside.std())/255.)
    assert set(flat)==set(EXTRA_FIELDS),('Missing feature',set(EXTRA_FIELDS)-set(flat))
    return flat

def extended_features(stem,masks,origins=None,curve_meta=None,gray=None,raw=None):
    if not masks:return pd.DataFrame(columns=NEW_FEATURES)
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
    if raw is None:raw=load_base(stem)[0]
    # Refinement-based V24 features need the frozen, correct V24 refiner.
    if origins is None:origins=['detector']*len(masks)
    if curve_meta is None:curve_meta=[{} for _ in masks]
    assert len(masks)==len(origins)==len(curve_meta)
    refined=v24_refined_masks(stem,masks,gray) if all(o=='detector' for o in origins) else [
        run_refiner(refiner_v24,gray,r) if o=='detector' else r for r,o in zip(masks,origins)]
    base=features_v24(stem,masks,refined,gray,raw)
    boxes=[list(map(float,coco.toBbox(rr))) for rr in masks]
    for i,rr in enumerate(masks):
        extra=extract_extra(gray,rr,boxes,origins[i],curve_meta[i],raw)
        for k in EXTRA_FIELDS:base.loc[i,k]=extra[k]
    return base[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)

print('Extended feature count:',len(NEW_FEATURES))

## 11. Cross-fitted quality estimator, with an acquisition-day audit

Fit on 53 historic held-out detector scenes, evaluate the new model on 51 different scenes. Within the 53-scene calibration portion, use 4-fold **group out-of-fold predictions** (not in-sample probabilities) to choose the instance-quality cutoff. Rank-average two deliberately different models. Training samples include both frozen detector proposals and curve proposals. The same historical 104 scenes were explored in V20–V24, so this remains exploratory and is **not** an independently sealed competition score.

In [ ]:
VAL_STEMS=sorted(VALIDATION)
assert len(VAL_STEMS)==104 or not FULL_VALIDATION_REQUIRED
by_day=defaultdict(list)
for st in VAL_STEMS:by_day[st[:8]].append(st)
all_days=sorted(by_day);random.Random(SEED+952).shuffle(all_days)
CAL_DAYS=set(all_days[:int(len(all_days)*.52)])
CAL=sorted(s for s in VAL_STEMS if s[:8] in CAL_DAYS)
AUDIT=sorted(s for s in VAL_STEMS if s[:8] not in CAL_DAYS)
assert len(CAL)>30 and len(AUDIT)>30 and not ({s[:8] for s in CAL}&{s[:8] for s in AUDIT})
print('Train quality on',len(CAL),'scenes, evaluate on',len(AUDIT),'different-day scenes')

# Cache masks and features separately from the checkpointed models.
def scene_candidates(stem,include_curve=True):
    raw,base=load_base(stem)
    if include_curve:
        full,origins=extend_with_curve(stem,base,raw)
    else:full,origins=base,['detector']*len(base)
    curve_list=curve_proposals(stem) if include_curve else []
    meta=[]
    for index,rr in enumerate(full):
        if origins[index]=='detector':meta.append({})
        else:
            index_ious=np.asarray(coco.iou([rr],[as_rle(v['counts']) for v in curve_list],
                                             [0]*len(curve_list)),float).reshape(-1) if curve_list else np.asarray([])
            meta.append(curve_list[int(index_ious.argmax())] if len(index_ious) else {})
    return raw,full,origins,meta,len(base)

def scene_feature_data(stem):
    path=OUT/'reports'/('features_'+stem+'.joblib')
    # Data are caches of MODEL PREDICTIONS, not ground-truth test labels.
    signature=hashlib.sha256((CURVE_SHA+REAL_REFINER_SHA+sha(V24_REFINER_PATH)+str(NEW_FEATURES)).encode()).hexdigest()
    if path.exists():
        try:
            obj=joblib.load(path)
            if obj.get('signature')==signature:return obj
        except Exception:pass
    raw,masks,origins,meta,n_base=scene_candidates(stem)
    feats=extended_features(stem,masks,origins,meta,raw=raw)
    obj={'signature':signature,'masks':[txt(x) for x in masks], 'origins':origins,
         'n_base':n_base,'features':feats,'stem':stem}
    joblib.dump(obj,path,compress=3)
    return obj

def consensus_quality_labels(stem,preds):
    expert_versions=gt(stem);out=[]
    for rr in preds:
        votes=[]
        for truth in expert_versions:
            if truth:
                vec=np.asarray(coco.iou([rr],truth,[0]*len(truth)),float).reshape(-1)
                votes.append(float(np.max(vec)) if len(vec) else 0.)
            else:votes.append(0.)
        out.append(int(np.mean(np.asarray(votes)>.5)>=.5) if votes else 0)
    return out

quality_train_features=[];quality_train_targets=[];quality_train_groups=[];quality_train_tags=[]
for stem in tqdm(CAL,desc='V25 quality training examples (no audit labels)'):
    obj=scene_feature_data(stem)
    masks=[as_rle(c) for c in obj['masks']]
    y=consensus_quality_labels(stem,masks)
    assert len(y)==len(obj['features'])
    quality_train_features.append(obj['features'])
    quality_train_targets.extend(y)
    quality_train_groups.extend([stem[:8]]*len(y))
    quality_train_tags.extend(obj['origins'])
X=pd.concat(quality_train_features,ignore_index=True)[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)
y=np.asarray(quality_train_targets,np.int64)
groups=np.asarray(quality_train_groups)
assert len(y)>180 and len(set(y))==2 and len(set(groups))>=4
print('Train instances',len(y),'positive fraction',round(float(y.mean()),3),'curve proposals',quality_train_tags.count('curve'))

def make_rankers():
    return [HistGradientBoostingClassifier(max_iter=140,max_leaf_nodes=11,min_samples_leaf=16,
                  learning_rate=.045,l2_regularization=7.,random_state=SEED),
            ExtraTreesClassifier(n_estimators=240,max_depth=12,min_samples_leaf=4,
                  max_features=.72,class_weight='balanced',n_jobs=-1,random_state=SEED)]

def rank_probs(models,df):
    if df.empty:return np.asarray([],float)
    a=df[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)
    return np.mean([model.predict_proba(a)[:,1] for model in models],axis=0)

folds=GroupKFold(n_splits=4)
oof_prob=np.zeros(len(y),float)
for k,(tr,va) in enumerate(folds.split(X,y,groups)):
    if len(np.unique(y[tr]))<2:raise ValueError('OOF fold has only one positive class')
    models=make_rankers()
    for model in models:model.fit(X.iloc[tr],y[tr])
    oof_prob[va]=rank_probs(models,X.iloc[va])
    print('Quality group OOF fold',k+1,'size',len(va))
new_quality_auc=float(roc_auc_score(y,oof_prob))
print('V25 group OOF AUROC',round(new_quality_auc,4))
# Conservative fixed grid, tuned solely using CAL OOF predictions, *never* the audit set.
def choose_oof_threshold():
    # PQ is pooled by scene and expert, not generic classification accuracy.
    choices=[.18,.22,.26,.30,.34,.38,.42,.46]
    # Build per scene OOF selection using the exact row sequence above.
    lookup={};start=0
    for stem in CAL:
        obj=scene_feature_data(stem);n=len(obj['masks']);lookup[stem]=(start,start+n);start+=n
    scores={}
    for th in choices:
        counts=np.zeros(4,float)
        for stem,(a,b) in lookup.items():
            masks=[as_rle(z) for z,p in zip(scene_feature_data(stem)['masks'],oof_prob[a:b]) if p>=th]
            counts+=physical_counts(stem,masks)
        scores[th]=pq(counts)
    # Prefer more conservative threshold if scores tie.
    threshold=max(scores,key=lambda t:(scores[t],t))
    return float(threshold),scores
V25_QUALITY_CUTOFF,OOF_PQ=choose_oof_threshold()
print('OOF-tuned quality threshold',V25_QUALITY_CUTOFF,'CAL OOF PQ',OOF_PQ)
new_rankers=make_rankers()
for model in new_rankers:model.fit(X,y)
V25_QUALITY_PATH=Path('/kaggle/working/SOLAR_V25_EXTENDED_QUALITY.joblib')
joblib.dump({'models':new_rankers,'columns':NEW_FEATURES,'cutoff':V25_QUALITY_CUTOFF,
 'oof_auc':new_quality_auc,'calibration_days':sorted(CAL_DAYS),'expert_consensus':True,
 'training_detector_proposals_from_held_out_scenes':True,'test_labels_used':False},V25_QUALITY_PATH)
save_json(OUT/'reports'/'V25_ranker_validation.json',{
 'train_instances':len(y),'feature_count':len(NEW_FEATURES),'new_curve_proposals':quality_train_tags.count('curve'),
 'group_oof_auc':new_quality_auc,'oof_PQ_by_threshold':OOF_PQ,'chosen_quality_cutoff':V25_QUALITY_CUTOFF,
 'calibration_scenes':len(CAL),'audit_scenes':len(AUDIT),
 'warning':'Original V20 holdout was reused by earlier experiments; not an untouched test set.'})

## 12. Train a **fragment-link classifier**, gated by independently measured PQ

Use potential filament fragment pairs within short endpoint/box distances. The binary label requires both fragments to correspond to the **same annotated physical instance** within an expert annotation, without access to test annotations. We train only on quality-calibration scenes. If there are insufficient positive links, the graph branch explicitly disables itself (rather than inventing a graph model).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Endpoint approximation by principal axis extremes of instance pixels, avoiding 2048x2048 skeleton overhead.
def pair_descriptor(rr,gray):
    x,y,w,h=map(float,coco.toBbox(rr));ix=int(x);iy=int(y)
    mw=int(np.ceil(w));mh=int(np.ceil(h))
    roi=coco.decode(rr)[iy:min(H,iy+mh),ix:min(W,ix+mw)].astype(np.uint8)
    sy,sx=np.where(roi>0)
    if len(sx)<4:
        pts=np.array([[x,y],[x+w,y+h]],float)
        return pts,np.array([1.,0.]),float(coco.area(rr))
    coords=np.stack([sx+ix,sy+iy],axis=1).astype(float)
    if len(coords)>1200:coords=coords[::max(1,len(coords)//1200)]
    eigvals,eigvecs=np.linalg.eigh(np.cov(coords.T));direction=eigvecs[:,np.argmax(eigvals)]
    projection=coords@direction
    pts=np.stack([coords[int(np.argmin(projection))],coords[int(np.argmax(projection))]])
    return pts,direction,float(coco.area(rr))

def graph_pair_features(d1,d2,gray):
    points_a,vec_a,area_a=d1;points_b,vec_b,area_b=d2
    d=np.linalg.norm(points_a[:,None,:]-points_b[None,:,:],axis=2)
    ai,bi=np.unravel_index(np.argmin(d),d.shape);dist=float(d[ai,bi])
    if dist>170:return None
    a=points_a[ai];b=points_b[bi]
    n=max(2,int(dist/4));xs=np.clip(np.rint(np.linspace(a[0],b[0],n)).astype(int),0,W-1)
    ys=np.clip(np.rint(np.linspace(a[1],b[1],n)).astype(int),0,H-1)
    bridge=gray[ys,xs].astype(float)
    link=(b-a)/(np.linalg.norm(b-a)+1e-8)
    return [dist/2048.,float(np.abs(vec_a@vec_b)),float(np.abs(vec_a@link)),
        float(np.abs(vec_b@link)),float(np.log1p(area_a)),float(np.log1p(area_b)),
        float(np.abs(np.log((area_a+1)/(area_b+1)))),float(bridge.mean()/255.),
        float(bridge.std()/255.),float(np.percentile(bridge,25)/255.),
        float(np.hypot(*(a-b))/max(np.sqrt(area_a+area_b),1.))]

GRAPH_COLUMNS=['endpoint_distance','alignment','link_approach_a','link_approach_b','log_area_a',
 'log_area_b','log_area_difference','bridge_intensity','bridge_std','bridge_p25','normalized_gap']

def positive_identity(stem,rr):
    # Each expert receives its own identity namespace.
    best=None;highest=0.
    for k,expert in enumerate(gt(stem)):
        if not expert:continue
        overlap=np.asarray(coco.iou([rr],expert,[0]*len(expert)),float).reshape(-1)
        idx=int(overlap.argmax())
        if overlap[idx]>highest:highest=float(overlap[idx]);best=(k,idx)
    return best if highest>.16 else None

def possible_graph_pairs(stem,masks,limit=110):
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
    descriptors=[pair_descriptor(r,gray) for r in masks]
    pairs=[]
    for i in range(len(masks)):
        for j in range(i+1,len(masks)):
            if len(pairs)>=limit:break
            if float(coco.iou([masks[i]],[masks[j]],[0])[0,0])>.18:continue
            feature=graph_pair_features(descriptors[i],descriptors[j],gray)
            if feature is not None:pairs.append((i,j,feature))
    return pairs

graph_X=[];graph_y=[];graph_g=[]
for stem in tqdm(CAL,desc='Learn graph links from calibration scenes'):
    obj=scene_feature_data(stem)
    masks=[as_rle(s) for s in obj['masks']]
    ids=[positive_identity(stem,r) for r in masks]
    for i,j,feature in possible_graph_pairs(stem,masks):
        graph_X.append(feature)
        graph_y.append(int(ids[i] is not None and ids[i]==ids[j]))
        graph_g.append(stem[:8])
graph_y=np.asarray(graph_y,int)
GRAPH_ACTIVE=bool(RUN_GRAPH and len(graph_X)>=70 and graph_y.sum()>=10 and
                  (len(graph_y)-graph_y.sum())>=20)
GRAPH_PATH=Path('/kaggle/working/SOLAR_V25_GRAPH_LINKS.joblib')
if GRAPH_ACTIVE:
    graph_model=RandomForestClassifier(n_estimators=250,max_depth=7,min_samples_leaf=3,
                     class_weight='balanced',random_state=SEED,n_jobs=-1)
    graph_model.fit(np.asarray(graph_X,float),graph_y)
    joblib.dump({'model':graph_model,'features':GRAPH_COLUMNS,'positives':int(graph_y.sum()),
                 'n_pairs':len(graph_y),'calibration_days_only':True},GRAPH_PATH)
    print('Learned graph:',len(graph_y),'pairs,',int(graph_y.sum()),'positive same-filament links')
else:
    graph_model=None
    save_json(OUT/'reports'/'graph_disabled.json',{'reason':'Too few positive and negative expert-supported pairs',
         'pairs':len(graph_y),'positive':int(graph_y.sum())})
    print('Graph safely disabled: insufficient supervised link examples')

def graph_merge(stem,masks,quality_prob,threshold=.95):
    if not GRAPH_ACTIVE or len(masks)<2:return masks
    pairs=possible_graph_pairs(stem,masks)
    if not pairs:return masks
    p=graph_model.predict_proba(np.asarray([q[2] for q in pairs],float))[:,1]
    # Only high-confidence proposals are considered for possible union.
    parent=list(range(len(masks)))
    def root(i):
        while parent[i]!=i:i=parent[i]
        return i
    for (i,j,_),prob in sorted(zip(pairs,p),key=lambda t:-t[1]):
        if prob<threshold or quality_prob[i]<.42 or quality_prob[j]<.42:continue
        a,b=root(i),root(j)
        if a==b:continue
        # Two fragments, not an arbitrary growing connected component.
        if sum(root(z)==a for z in range(len(masks)))+sum(root(z)==b for z in range(len(masks)))>2:continue
        parent[b]=a
    groups=defaultdict(list)
    for i in range(len(masks)):groups[root(i)].append(i)
    out=[]
    for indices in groups.values():
        out.append(coco.merge([masks[i] for i in indices]) if len(indices)>1 else masks[indices[0]])
    return out

## 13. Ablations against V24 public-0.38 control on distinct audit days

Each candidate represents one hypothesis. The new quality threshold was chosen using **calibration OOF**, and all comparisons below use the other 51 observation scenes. An improved local score does not prove a better Kaggle public score. In particular the graph component is disabled if same-instance fragment labels are inadequate.

In [ ]:
# Explicit, separately scoreable hypotheses. All use the trained models, not only threshold sweeps.
ABLATIONS={
 'V24_PUBLIC_038_CONTROL':{'use_v24':True},
 'V25_EXTENDED_QUALITY':{'curve':False,'real_refine':False,'graph':False},
 'V25_CURVE_PLUS_QUALITY':{'curve':True,'real_refine':False,'graph':False},
 'V25_REAL_REFINE_PLUS_QUALITY':{'curve':False,'real_refine':True,'graph':False},
 'V25_GRAPH_PLUS_QUALITY':{'curve':False,'real_refine':False,'graph':True},
 'V25_CURVE_GRAPH':{'curve':True,'real_refine':False,'graph':True},
 'V25_SELECTIVE_ALL':{'curve':True,'real_refine':True,'graph':True},
 'V25_CONSERVATIVE_CURVE':{'curve':True,'real_refine':False,'graph':False,'quality_margin':.08},
}

def nonoverlap_after_refinement(mask_sequence):
    occupied=np.zeros((H,W),np.uint8);out=[]
    for rr in mask_sequence:
        pix=coco.decode(rr).astype(np.uint8)&(occupied==0)
        if int(pix.sum())<80:continue
        out.append(enc(pix));occupied|=pix
    return out

def output_candidate(stem,name,info=None,probabilities=None):
    cfg=ABLATIONS[name]
    if cfg.get('use_v24'):return v24_control(stem)
    info=info if info is not None else scene_feature_data(stem)
    all_masks=[as_rle(c) for c in info['masks']]
    p=probabilities if probabilities is not None else rank_probs(new_rankers,info['features'])
    assert len(all_masks)==len(info['origins'])==len(p)
    cutoff=V25_QUALITY_CUTOFF+cfg.get('quality_margin',0.)
    allowed=set(range(len(all_masks))) if cfg.get('curve') else set(range(info['n_base']))
    chosen_indices=[i for i in range(len(all_masks)) if i in allowed and p[i]>=cutoff]
    result=[all_masks[i] for i in chosen_indices]
    q=[p[i] for i in chosen_indices]
    if cfg.get('real_refine') and result:
        gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
        result=[real_refine(stem,rr,gray) if info['origins'][i]=='detector' and p[i]>=.50 else rr
                for i,rr in zip(chosen_indices,result)]
        result=nonoverlap_after_refinement(result)
    if cfg.get('graph') and result and len(q)==len(result):
        result=graph_merge(stem,result,q,threshold=.95)
        result=nonoverlap_after_refinement(result)
    return result

# Evaluate 51 days only. CAL data is for training and OOF tuning, NOT used to choose among ablations.
AUDIT_COUNTS={name:np.zeros(4,float) for name in ABLATIONS}
EVAL_RECORDS=[]
for stem in tqdm(AUDIT,desc='Independent-day V25 ablations'):
    info=scene_feature_data(stem)
    probabilities=rank_probs(new_rankers,info['features'])
    for name in ABLATIONS:
        masks=output_candidate(stem,name,info,probabilities)
        counts=physical_counts(stem,masks)
        AUDIT_COUNTS[name]+=counts
        EVAL_RECORDS.append({'stem':stem,'day':stem[:8],'method':name,'sum_iou':float(counts[0]),
           'TP':int(counts[1]),'FP':int(counts[2]),'FN':int(counts[3]),'n_instances':len(masks)})
control='V24_PUBLIC_038_CONTROL'
control_audit=pq(AUDIT_COUNTS[control])
leader=pd.DataFrame([{'method':name,**metrics(vals),'audit_delta_vs_V24':pq(vals)-control_audit,
                      'curve_added':bool(ABLATIONS[name].get('curve',False)),
                      'graph_used':bool(ABLATIONS[name].get('graph',False) and GRAPH_ACTIVE),
                      'trained_refiner_used':bool(ABLATIONS[name].get('real_refine',False))}
                     for name,vals in AUDIT_COUNTS.items()]).sort_values('PQ',ascending=False)
leader.to_csv(OUT/'reports'/'V25_AUDIT_PQ.csv',index=False)
pd.DataFrame(EVAL_RECORDS).to_csv(OUT/'reports'/'V25_AUDIT_PER_IMAGE.csv',index=False)
display(leader)

# Paired acquisition-day bootstrap: estimate audit uncertainty, NOT leaderboard score.
day_matrix=pd.DataFrame(EVAL_RECORDS).groupby(['method','day'])[['sum_iou','TP','FP','FN']].sum()
audit_days=sorted({r['day'] for r in EVAL_RECORDS})
mat={name:np.stack([day_matrix.loc[(name,day)].to_numpy(float) for day in audit_days]) for name in ABLATIONS}
rng=np.random.default_rng(SEED+25)
ix=rng.integers(0,len(audit_days),size=(1100,len(audit_days)))

def batch_pq(values):
    d=values[:,1]+.5*(values[:,2]+values[:,3])
    return np.divide(values[:,0],d,out=np.zeros(len(values)),where=d>0)
base_boot=batch_pq(mat[control][ix].sum(axis=1))
boot_rows=[]
for name in ABLATIONS:
    if name==control:continue
    delta=batch_pq(mat[name][ix].sum(axis=1))-base_boot
    boot_rows.append({'method':name,'delta_audit_PQ':pq(AUDIT_COUNTS[name])-control_audit,
         'delta_2p5':float(np.quantile(delta,.025)),'delta_97p5':float(np.quantile(delta,.975)),
         'fraction_bootstrap_positive':float(np.mean(delta>0))})
boot_report=pd.DataFrame(boot_rows).sort_values('delta_audit_PQ',ascending=False)
boot_report.to_csv(OUT/'reports'/'V25_PAIRED_DAY_BOOTSTRAP.csv',index=False)
display(boot_report)

# Gate avoids automatically submitting a method that barely wins an exhausted proxy.
new_rows=leader[(leader.method!=control)&(leader.audit_delta_vs_V24>=.005)]
SELECTED=control if new_rows.empty else str(new_rows.iloc[0].method)
print('Conservative default:',SELECTED,'(new public score unknown)')
save_json(OUT/'reports'/'V25_method_selection.json',{
 'default':SELECTED,'verified_historical_public_PQ':.38,'new_public_PQ':None,
 'quality_cutoff_tuned_only_on_calibration_OOF':V25_QUALITY_CUTOFF,
 'historical_V20_validation_set_reuse':True,'audit_scenes':len(AUDIT),
 'audit_PQ_by_method':{name:pq(v) for name,v in AUDIT_COUNTS.items()},
 'graph_active':GRAPH_ACTIVE,'official_private_evaluator_parity_verified':False})

## 14. Full test inference, untouched proven-control recovery, and separate CSV exports

All 180 hidden-label images are processed. Exports contain COCO compressed RLE and `filament_id`, exactly as in the successfully scored V24 notebook. A SHA-256 check distinguishes the exact **previously submitted 0.38 bytes** from a merely reconstructed baseline. Any branch that performs worse locally remains available as an experiment but cannot silently become the known winner.

In [ ]:
from contextlib import ExitStack
TEST_STEMS=sorted(TEST)
assert len(TEST_STEMS)==180
METHODS=list(ABLATIONS)
SUBROOT=OUT/'submissions'
for stem in tqdm(TEST_STEMS,desc='Generate all real test predictions (V24 + V25)'):
    scene_feature_data(stem)  # creates detector & curve masks and caches 60+ features

# Keep stream writes deterministic; avoid materializing thousands of full masks at once.
SUBMISSION_PATHS={name:SUBROOT/(name+'.csv') for name in METHODS}
image_count_records=[]
with ExitStack() as stack:
    handles={name:stack.enter_context(open(path,'w',encoding='utf8'))
             for name,path in SUBMISSION_PATHS.items()}
    for handle in handles.values():handle.write('filament_id,segmentation_rle\n')
    for stem in tqdm(TEST_STEMS,desc='Encode V25 candidate CSVs'):
        info=scene_feature_data(stem)
        proba=rank_probs(new_rankers,info['features'])
        for name in METHODS:
            masks=output_candidate(stem,name,info,proba)
            image_count_records.append({'stem':stem,'method':name,'instances':len(masks)})
            for index,rr in enumerate(masks,1):
                counts=txt(rr)
                assert ',' not in counts and '\n' not in counts
                handles[name].write(f'{stem}_{index},{counts}\n')

manifest={};count_df=pd.DataFrame(image_count_records)
count_df.to_csv(OUT/'reports'/'V25_TEST_INSTANCE_COUNTS.csv',index=False)
for name,path in SUBMISSION_PATHS.items():
    frame=pd.read_csv(path,dtype=str)
    assert frame.columns.tolist()==['filament_id','segmentation_rle']
    assert len(frame)>200 and frame.filament_id.is_unique
    assert frame.segmentation_rle.notna().all()
    assert frame.filament_id.str.rsplit('_',n=1).str[0].isin(TEST).all()
    count_slice=count_df[count_df.method==name]
    assert len(count_slice)==180 and count_slice.stem.nunique()==180
    assert int(count_slice.instances.sum())==len(frame)
    for text_rle in frame.segmentation_rle:
        assert int(coco.area(as_rle(text_rle)))>0
    manifest[name]={'rows':len(frame),'sha256':sha(path),'path':str(path),'public_PQ':None}
    print(name,'rows',len(frame),'SHA',sha(path)[:18])

historical=SUBMISSION_PATHS[control]
EXACT_HISTORY=sha(historical).startswith(V24_EXPECTED_REFERENCE_SHA)
BASELINE_COPY=Path('/kaggle/working/V25_PROTECTED_V24_038.csv')
shutil.copy2(historical,BASELINE_COPY)
if not EXACT_HISTORY:
    payload,_=find_original_asset('V24_V24_QUALITY_025.csv')
    if payload is not None and hashlib.sha256(payload).hexdigest().startswith(V24_EXPECTED_REFERENCE_SHA):
        BASELINE_COPY.write_bytes(payload);EXACT_HISTORY=True
    else:
        payload,_=find_original_asset('submission.csv')
        if payload is not None and hashlib.sha256(payload).hexdigest().startswith(V24_EXPECTED_REFERENCE_SHA):
            BASELINE_COPY.write_bytes(payload);EXACT_HISTORY=True
if EXACT_HISTORY:print('Exact historic public-0.38 CSV SHA recovered.')
else:
    print('WARNING: V24 method was regenerated, but exact previous leaderboard CSV SHA did not match.')

# Do not present the unverified generated control as an exact known-public file.
DEFAULT_SOURCE=BASELINE_COPY if SELECTED==control else SUBMISSION_PATHS[SELECTED]
shutil.copy2(DEFAULT_SOURCE,Path('/kaggle/working/submission.csv'))
for i,name in enumerate([x for x in METHODS if x!=control],1):
    shutil.copy2(SUBMISSION_PATHS[name],Path('/kaggle/working')/f'V25_ALT_{i}.csv')
manifest['V24_HISTORICAL_CONTROL_SHA_MATCH']=EXACT_HISTORY
save_json(OUT/'reports'/'V25_submission_manifest.json',{
 'candidates':manifest,'default_method':SELECTED,
 'historical_038_exact_bytes':EXACT_HISTORY,
 'default_source':str(DEFAULT_SOURCE),'new_hidden_public_score':None,
 'test_labels_read':False,'trained_curve_SHA256':CURVE_SHA,
 'trained_real_refiner_SHA256':REAL_REFINER_SHA,
 'trained_quality_model_SHA256':sha(V25_QUALITY_PATH),
 'graph_trained':GRAPH_ACTIVE})
print('Final submission.csv:',SELECTED,' | preserved prior-public-0.38 at:',BASELINE_COPY)

## 15. Reproducibility bundles: all models, output CSVs and research ablations

Two output archives: a lightweight CSV+reports ZIP and a full checkpoint bundle with V20, V24 and V25 model bytes. The full model ZIP is essential for resuming without retraining; it should be saved as a Kaggle Dataset after Run All. Intermediate model checkpoint files are created *before* test inference, so they can be preserved even if the last cells are interrupted.

In [ ]:
REPORT_ZIP=Path('/kaggle/working/SOLAR_V25_ALL_PREDICTIONS_AND_REPORTS.zip')
with zipfile.ZipFile(REPORT_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3,allowZip64=True) as z:
    for file in sorted((OUT/'reports').glob('*.csv'))+sorted((OUT/'reports').glob('*.json')):
        z.write(file,arcname='reports/'+file.name)
    for file in sorted(SUBROOT.glob('*.csv')):
        z.write(file,arcname='submissions/'+file.name)
    z.write('/kaggle/working/submission.csv',arcname='submission.csv')
    z.write(BASELINE_COPY,arcname=BASELINE_COPY.name)
with zipfile.ZipFile(REPORT_ZIP) as z:
    assert z.testzip() is None
    assert 'submission.csv' in z.namelist()

FULL_ZIP=Path('/kaggle/working/SOLAR_V25_ALL_TRAINED_CHECKPOINTS.zip')
with zipfile.ZipFile(FULL_ZIP,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    for name,p in WEIGHTS.items():z.write(p,arcname='detectors/'+p.name)
    for p in [V24_REFINER_PATH,V24_QUALITY_PATH,curve_path,curve_last,real_best_path,real_last_path,V25_QUALITY_PATH]:
        if p.is_file():z.write(p,arcname='models/'+p.name)
    if GRAPH_ACTIVE:z.write(GRAPH_PATH,arcname='models/'+GRAPH_PATH.name)
    z.write(REPORT_ZIP,arcname=REPORT_ZIP.name)
with zipfile.ZipFile(FULL_ZIP) as z:assert z.testzip() is None
print('DONE: 0.38 control kept. New public scores UNKNOWN.')
print('Submission:',Path('/kaggle/working/submission.csv'))
print('All predictions and reports:',REPORT_ZIP,'MB',round(REPORT_ZIP.stat().st_size/2**20,2))
print('Full reusable trained models:',FULL_ZIP,'MB',round(FULL_ZIP.stat().st_size/2**20,2))

## What to upload and how to interpret results

- Submit `submission.csv` first **only if it differs from the exact V24 0.38 control**; the primary label is shown in `V25_method_selection.json`. Additional experimental files are `V25_ALT_1.csv` ... `V25_ALT_7.csv` and carry recipe names in the ZIP.
- Check `V25_AUDIT_PQ.csv`, `V25_PAIRED_DAY_BOOTSTRAP.csv`, `V25_ranker_validation.json`, and `V25_submission_manifest.json` before trusting any gain. A positive local difference does not establish leaderboard improvement.
- The curve head, real-mask refiner, extended ranking ensemble and graph learner have **separate ablations**. If a graph is unsupported by enough supervised link examples it is disabled and reported, not silently used.
- For stricter research results, train fresh detector folds on training-only scene groups and gather genuine detector OOF predictions; this expensive optional extension is not executed by this notebook. Its 4-fold cross-fitting refers specifically to the **quality ranker**, not detector cross-fitting. The reused 104-image V20 tune set and leaderboard feedback have influenced our choices.
- Do **not** use publicly exposed test annotations, external test-image label matching, or evaluation exploits. The benchmark should reflect models' segmentation ability on supplied images.